# Variance formula diagnostics

Are the null variances used by the sequential detector actually correct?

The test fires when $\mathrm{Sus}(t) > c\,\bar\sigma(t)/\sqrt{t}$, so every claim
about false-alarm rates rests on $\sigma^{2,(t)}$ being the true null variance of
the per-round score. This notebook checks that claim three independent ways:

| # | Check | Method |
|---|-------|--------|
| A | Is the score mean-zero under the null? | empirical, across sims |
| B | Does $\sigma^2$ match the empirical per-round variance? | across sims, at each round |
| C | Does $\sigma^2$ match the variance of a **replayed** round? | freeze one state, resample it |
| D | Does $\mathrm{Var}[\mathrm{Sus}(t)] = \bar\sigma^2(t)/t$ hold? | empirical + autocorrelation |

Check C is the decisive one. B has a confound — at round $t$ different sims sit in
*different listener states*, so it pools things with genuinely different variances. C
removes that by freezing a single state and replaying that round many times, which is a
direct measurement of what $\sigma^2$ claims to be.

**No re-simulation is required for A, B, or D**: `results/full_sweep/trajectories/`
already stores the per-round score *and* its claimed variance. C needs live
`ScoreContext` objects, so it runs short simulations inline (~30 s).

## 0. What the formulas claim

For a score $s^{(t)}$ with claimed null variance $\sigma^{2,(t)}$ the test accumulates

$$\mathrm{Sus}(t)=\frac1t\sum_{i\le t}s^{(i)},\qquad
\bar\sigma^2(t)=\frac1t\sum_{i\le t}\sigma^{2,(i)},\qquad
\text{fire when }\mathrm{Sus}(t)>c\,\bar\sigma(t)/\sqrt t .$$

That rule is valid only if all of the following hold under the null $\psi=\inf$:

1. $\mathbb{E}[s^{(t)}]=0$
2. $\mathrm{Var}[s^{(t)}]=\sigma^{2,(t)}$
3. the $s^{(t)}$ are (near) uncorrelated across $t$, which is what licenses the
   $1/t$ shrinkage in step 2 of the accumulation.

**surp2.** $s(u)=-\log P_{\text{prior}}(u)-H(P_{\text{prior}})$ with
$\sigma^2=\mathrm{Var}_{u\sim P_{\text{prior}}}[-\log P_{\text{prior}}(u)]$.
Here $\sigma^2$ is by construction the exact variance of $s$ under
$u\sim P_{\text{prior}}$ — this one should pass check C identically.

**sus_1.** $s(u)=\sum_O L_1(O\mid u)\,B(O,u)$ with
$B(O,u)=-\log S_1(u\mid O,\inf)-H(S_1(\cdot\mid O,\inf))$. Two candidate variances:

$$\sigma^2_{\text{naive}}=\sum_O L_1(O\mid u)\,\mathrm{varent}(O),\qquad
\sigma^2_{\text{corr}}=\sigma^2_{\text{naive}}-\!\!\sum_{u'}P_{\text{pred}}(u')\,
\mathrm{Var}_{O\sim W(\cdot\mid u')}[B(O,u')].$$

Note the correction *subtracts*, so $\sigma^2_{\text{corr}}\le\sigma^2_{\text{naive}}$
always. Which of the two — if either — is the real null variance of $s(u)$ is what
check C settles.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
SWEEP, GROUP1 = ROOT/"results/full_sweep", ROOT/"results/group1"

# Categorical slots 1-4 of the validated reference palette, in fixed order.
C = {"naive": "#2a78d6", "corrected": "#eb6834", "exact": "#1baf7a", "surp2": "#eda100"}
INK, INK2, GRIDC = "#0b0b0b", "#52514e", "#dcdcd8"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.titlesize": 9.5,
    "axes.edgecolor": GRIDC, "axes.labelcolor": INK2,
    "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.6,
    "xtick.color": INK2, "ytick.color": INK2,
    "legend.frameon": False, "lines.linewidth": 1.6,
})
DIVERGE = LinearSegmentedColormap.from_list("ratio", [C["naive"], "#e8e8e4", C["corrected"]])
SEQ     = LinearSegmentedColormap.from_list("mag", ["#f4f4f1", C["naive"]])

def ref_line(ax, y=1.0, label="perfect calibration"):
    ax.axhline(y, color=INK, lw=1.0, ls=(0, (4, 3)), zorder=1, label=label)

## 1. What data already exists

In [ ]:
sims = pd.concat([pd.read_parquet(p) for p in sorted((SWEEP/"simulations").glob("cell=*/part.parquet"))])
CELLS = sims[["cell_id", "theta_star", "psi_true", "alpha"]].drop_duplicates().reset_index(drop=True)
NULL  = CELLS[CELLS.psi_true == "inf"].reset_index(drop=True)
CFG   = json.loads((SWEEP/"run_config.json").read_text())

SCORE_COLS = ["round", "sim_id", "surp2_score", "surp2_sigma2",
              "sus1_score", "sus1_sigma2_naive", "sus1_sigma2_corrected"]

def load_cell(cell_id, cols=SCORE_COLS):
    return pd.read_parquet(SWEEP/f"trajectories/cell={int(cell_id):04d}/part.parquet", columns=cols)

def cell_id_for(theta, alpha, psi="inf"):
    m = CELLS[(CELLS.theta_star == theta) & (CELLS.alpha == alpha) & (CELLS.psi_true == psi)]
    if not len(m):
        raise KeyError(f"no cell for theta={theta} alpha={alpha} psi={psi}")
    return int(m.cell_id.iloc[0])

print(f"cells total     : {len(CELLS)}   (null psi=inf: {len(NULL)})")
print(f"alphas          : {sorted(CELLS.alpha.unique())}")
print(f"thetas          : {sorted(CELLS.theta_star.unique())}")
print(f"sims per cell   : {sims.groupby('cell_id').size().unique()}")
print(f"rounds per sim  : {load_cell(NULL.cell_id.iloc[0])['round'].max()}")
print(f"null rows avail : {len(NULL) * 200 * 150:,}")

Every column needed for checks A, B and D is already on disk — per-round `score`
**and** the claimed `sigma2`, for both variance definitions. Nothing needs re-running.

`surp1` is absent from this sweep (it was dropped after Group 1), so §6 falls back to
`results/group1/raw_trajectories.parquet`, which does carry it.

## A. Is the score mean-zero under the null?

If $\mathbb{E}[s]\neq0$ the running mean drifts and the detector fires on the drift
rather than on persuasion. Checked at $\theta^\star=0.5,\ \alpha=3$ (the Group-1 config).

In [ ]:
cid = cell_id_for(0.5, 3.0)
d = load_cell(cid)
g = d.groupby("round").sus1_score
mz = pd.DataFrame({"mean": g.mean(), "se": g.std(ddof=1)/np.sqrt(g.size())})
mz["z"] = mz["mean"]/mz["se"]

fig, ax = plt.subplots(figsize=(7.2, 2.9))
ax.fill_between(mz.index, -2*mz.se, 2*mz.se, color=GRIDC, alpha=.7, lw=0, label="±2 SE")
ax.plot(mz.index, mz["mean"], color=C["naive"], label="mean sus_1 score")
ax.axhline(0, color=INK, lw=1.0, ls=(0, (4, 3)))
ax.set_xlabel("round"); ax.set_ylabel("mean score")
ax.set_title(f"A. Null score mean, θ*=0.5, α=3 ({d.sim_id.nunique()} sims)", color=INK, loc="left")
ax.legend(loc="upper right", ncols=2)
plt.tight_layout(); plt.show()

print(f"mean over all rounds : {mz['mean'].mean():+.5f}")
print(f"rounds with |z| > 2  : {(mz.z.abs() > 2).sum()} / {len(mz)}  (expect ~{0.05*len(mz):.0f} by chance)")
print(f"max |z|              : {mz.z.abs().max():.2f}")

**Verdict A — holds.** The mean sits on zero with no visible drift and the excursion
count is at chance. The mean-zero assumption is not the problem.

## B. Does $\sigma^2$ match the empirical per-round variance?

The comparison must be made **within a round**, pooling across sims: $\sigma^{2,(t)}$
is a *conditional* variance given the listener's round-$t$ state, and pooling across
rounds mixes states with genuinely different variances.

Ratio $=\mathrm{Var}_{\text{emp}}[s^{(t)}]\;/\;\mathbb{E}[\sigma^{2,(t)}]$. It should be 1.
Below 1 means the formula **over**states the variance (conservative, threshold too high).

In [ ]:
def calib_by_round(cell_id):
    d = load_cell(cell_id)
    rows = []
    for t, g in d.groupby("round"):
        vs, v2 = g.sus1_score.var(ddof=1), g.surp2_score.var(ddof=1)
        rows.append({"round": t,
                     "sus_1 naive":     vs/g.sus1_sigma2_naive.mean(),
                     "sus_1 corrected": vs/g.sus1_sigma2_corrected.mean(),
                     "surp2":           v2/g.surp2_sigma2.mean()})
    return pd.DataFrame(rows).set_index("round")

cb = calib_by_round(cid)

fig, ax = plt.subplots(figsize=(7.2, 3.2))
for k, col in [("sus_1 naive", "naive"), ("sus_1 corrected", "corrected"), ("surp2", "surp2")]:
    ax.plot(cb.index, cb[k].rolling(5, min_periods=1).mean(), color=C[col], label=k)
ref_line(ax)
ax.set_xlabel("round"); ax.set_ylabel("Var_emp / E[σ²]"); ax.set_ylim(0.4, 1.3)
ax.set_title("B. Per-round variance calibration, θ*=0.5, α=3 (5-round rolling mean)", color=INK, loc="left")
ax.legend(loc="lower right", ncols=2)
plt.tight_layout(); plt.show()

print(cb.loc[[1, 2, 5, 10, 20, 50, 100, 150]].round(3).to_string())

Two things are visible: a **transient over the first ~5 rounds** where all three are
badly calibrated, and a **persistent gap for `sus_1 naive`** that `corrected` closes.
The early transient matters — that is exactly when spurious crossings happen.

Now sweep it: the same ratio across all 99 null cells (9 θ × 11 α).

In [ ]:
rows = []
for _, r in NULL.iterrows():
    d = load_cell(r.cell_id)
    for t, g in d.groupby("round"):
        vs, v2 = g.sus1_score.var(ddof=1), g.surp2_score.var(ddof=1)
        rows.append((r.alpha, r.theta_star, t, g.sus1_sigma2_naive.mean(),
                     vs/g.sus1_sigma2_naive.mean(),
                     vs/g.sus1_sigma2_corrected.mean(),
                     v2/g.surp2_sigma2.mean()))
R = pd.DataFrame(rows, columns=["alpha", "theta", "round", "scale",
                                "naive", "corrected", "surp2"])
R.to_parquet(SWEEP/"variance_calibration.parquet")   # cache for re-use
late = R[R["round"] >= 20]           # drop the start-up transient

One trap has to be cleared first. At high α the speaker becomes near-deterministic and
the score collapses toward a constant — every sim emits the same utterance. The ratio
then becomes 0/0, a meaningless number that reads as "bad calibration" if left in.

Screen those out by the **empirical variance** of the score, never by the ratio itself
(that would be circular — excluding cells for having the outcome you are measuring).

In [ ]:
DEGEN_FLOOR = 1e-3                       # absolute floor on Var_emp[s]
late = late.assign(var_emp=late.naive*late.scale)
late = late.assign(degen=late.var_emp < DEGEN_FLOOR)

print(f"round-cells with Var_emp < {DEGEN_FLOOR:g} (score carries no signal): "
      f"{100*late.degen.mean():.1f}%\n")
print("degenerate share by α:")
print(late.groupby("alpha").degen.mean().round(3).to_string())

by_alpha = late[~late.degen].groupby("alpha")[["naive", "corrected", "surp2"]].median()
print("\nmedian calibration ratio by α (non-degenerate cells):")
print(by_alpha.round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

ax = axes[0]
for k in ["naive", "corrected", "surp2"]:
    lbl = {"naive": "sus_1 naive", "corrected": "sus_1 corrected", "surp2": "surp2"}[k]
    ax.plot(by_alpha.index, by_alpha[k], marker="o", ms=4, color=C[k], label=lbl)
ref_line(ax)
ax.set_xscale("log"); ax.set_xticks(by_alpha.index)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel("α (speaker rationality)"); ax.set_ylabel("Var_emp / E[σ²]")
ax.set_title("B1. Calibration vs α (median over θ, rounds ≥ 20)", color=INK, loc="left")
ax.legend(loc="lower center", ncols=2)

ax = axes[1]
piv = late.pivot_table(index="theta", columns="alpha", values="degen", aggfunc="mean")
im = ax.imshow(piv.values, cmap=SEQ, vmin=0, vmax=1, aspect="auto", origin="lower")
ax.set_xticks(range(len(piv.columns)), [f"{a:g}" for a in piv.columns])
ax.set_yticks(range(len(piv.index)), [f"{t:g}" for t in piv.index])
ax.set_xlabel("α"); ax.set_ylabel("θ*"); ax.grid(False)
ax.set_title("B2. Share of rounds where the score carries no signal", color=INK, loc="left")
fig.colorbar(im, ax=ax, label="fraction degenerate")
plt.tight_layout(); plt.show()

**Verdict B — `naive` is over-conservative at low α; `corrected` fixes it everywhere.**

- `surp2` is well calibrated throughout (≈ 0.96–1.01).
- `sus_1 naive` overstates the variance by ~30 % for α ≤ 3 (ratio 0.69–0.78) — a
  materially over-conservative per-round threshold. It recovers on its own by α ≈ 7.
- `sus_1 corrected` sits at 0.90–1.01 across the whole α range and is never worse
  than `naive`. Above α ≈ 10 the two coincide: the correction term has gone to zero.
- **There is no high-α calibration failure.** An earlier reading of these curves showed
  one, but it was entirely an artifact of degenerate cells; once they are excluded on
  `Var_emp` the ratio is flat.
- The real high-α problem is different and more basic: the score **stops carrying signal**.
  Zero degenerate rounds for α ≤ 5, then 10 % at α = 10, 27 % at α = 15, 34 % at α = 20.
  That is a limit on the detector, not on the variance formula.

## C. Replay one frozen state and measure the variance directly

§B has a confound. At round $t$, different sims are in **different listener states**, so
pooling their scores mixes states that genuinely have different variances. $\sigma^{2,(t)}$
is a *conditional* variance given one state — to test it properly, hold the state fixed.

So: run a sim to round $t$, **freeze the listener**, then replay that same round many
times with fresh random draws and take the variance of the resulting scores. That is a
direct measurement of the thing $\sigma^2$ claims to be, with no state-mixing.

One detail that turns out to matter: $\sigma^2$ **depends on the utterance drawn**,
$\sigma^2(u)$, while the conditional variance of the score is a single number for the
state. The quantity the test effectively averages over rounds is $\mathbb{E}_u[\sigma^2(u)]$,
so that — not $\sigma^2$ at one realized $u$ — is what must be compared against
$\mathrm{Var}_u[s(u)]$.

Two sampling laws are worth replaying under, and the difference between them is the
whole story of §B:

- the **model null** — $O\sim q_t(O)$ from the listener's *current belief*, then
  $u\sim k_t(u\mid O)$. This is what the $\sigma^2$ formulas are derived against.
- the **true sampling law** — $O\sim P(O\mid\theta^\star)$, then $u\sim k_t(u\mid O)$.
  This is what actually generated the data, and what the real false-alarm rate depends on.

In [ ]:
from rsa.setup import make_world, make_semantics
from rsa.speaker0 import Speaker0
from rsa.listener0 import Listener0
from rsa.speaker1 import Speaker1
from rsa.detection import (ScoreContext, DetectionListener, SequentialTest,
                           compute_surp2, SUS_VARIANT_FNS)
from rsa.detection.scores import _weighting_matrix, _p_pred

TS, sus1 = CFG["theta_space"], SUS_VARIANT_FNS["1"]

def freeze_state(theta, alpha, freeze_round, seed=0, n=1, m=7):
    # Advance one simulation to `freeze_round` and return the frozen agents + beliefs.
    import random as _r
    np.random.seed(seed); _r.seed(seed)
    world, sem = make_world(theta, n=n, m=m), make_semantics(n=n)
    s0 = Speaker0(TS, semantics=sem, world=world)
    l0 = Listener0(TS, s0, semantics=sem, world=world)
    s1 = Speaker1(TS, l0, semantics=sem, world=world, alpha=alpha, psi="inf")
    tests = [SequentialTest(compute_surp2, "surp2", c=float("inf")),
             SequentialTest(sus1, "sus_1", c=float("inf"))]
    L = DetectionListener(TS, ["inf", "high", "low"], s1, world, sem, tests=tests, alpha=alpha)
    for _ in range(1, freeze_round):
        obs = world.sample_obs(); u = s1.sample_utterance(obs)
        L.update(u); s1.update(obs)
    return world, sem, s1, L._l1_theta_array(), L._l0_theta_array()

def replay_at_state(theta, alpha, freeze_round=50, n_replays=50000, seed=0, n=1, m=7):
    # Freeze the listener, then resample this round `n_replays` times.
    world, sem, s1, l1, l0a = freeze_state(theta, alpha, freeze_round, seed, n, m)
    utts = list(sem.utterances)

    # score and claimed variances at each possible utterance (state held fixed)
    per_u = np.array([
        (*sus1(ScoreContext(TS, s1, world, sem, l1, u, l0_theta=l0a))[0:3],
         compute_surp2(ScoreContext(TS, s1, world, sem, l1, u, l0_theta=l0a))[1])
        for u in utts])
    s_u, vn_u, vc_u, v2_u = per_u.T

    ctx0 = ScoreContext(TS, s1, world, sem, l1, utts[0], l0_theta=l0a)
    p_model = np.asarray(_p_pred("1", ctx0), dtype=float)
    p_obs = world.obs_prob_table((theta,))[:, 0]; p_obs = p_obs/p_obs.sum()
    p_true = ctx0.S1_table.T @ p_obs; p_true = p_true/p_true.sum()

    out = {}
    for law, p in [("model", p_model), ("true", p_true)]:
        draws = np.random.choice(len(utts), size=n_replays, p=p)
        out[law] = {"mc_var": float(s_u[draws].var(ddof=1)),
                    "closed_form": float(p @ s_u**2 - (p @ s_u)**2),
                    "E_naive": float(p @ vn_u), "E_corrected": float(p @ vc_u)}
    out["tv_distance"] = float(0.5*np.abs(p_model - p_true).sum())
    return out

for alpha in [1.0, 3.0, 10.0]:
    o = replay_at_state(0.5, alpha, freeze_round=50)
    print(f"α={alpha:<5g} θ*=0.5, frozen at round 50, 50k replays "
          f"(‖p_model−p_true‖_TV = {o['tv_distance']:.3f})")
    for law in ["model", "true"]:
        r = o[law]
        print(f"   {law:5s} law: Var_replay={r['mc_var']:.5f}   "
              f"E[σ²_naive]={r['E_naive']:.5f} (ratio {r['mc_var']/r['E_naive']:.3f})   "
              f"E[σ²_corr]={r['E_corrected']:.5f} (ratio {r['mc_var']/r['E_corrected']:.3f})")

Under the **model null**, `corrected` lands on the replayed variance almost exactly,
while `naive` is ~10 % high at α = 1. Under the **true law** both drift a little,
because the listener's belief has not fully converged — that residual is the §B gap.

Notice also `Var_replay` ≈ `closed_form` in every row. That is not a coincidence, and it
is what the "closed form" in this notebook actually means: given the frozen state the
score is a deterministic function of $u$, and $u$ has only 8 possible values, so
sampling 50,000 times and summing 8 terms compute the same number. The closed form is
just the replay taken to its limit.

In [ ]:
o = replay_at_state(0.5, 1.0, freeze_round=50, n_replays=200000)["model"]
print(f"replay with 200k draws : {o['mc_var']:.6f}")
print(f"closed form (8 terms)  : {o['closed_form']:.6f}")
print(f"difference             : {abs(o['mc_var']-o['closed_form']):.2e}")

The same machinery settles `surp2` in one line. Its $\sigma^2$ is the varentropy of
$P_{\text{prior}}$, which *is* the variance of $-\log P_{\text{prior}}(u)$ under
$u\sim P_{\text{prior}}$ — so it should equal the closed form identically, not approximately.

In [ ]:
def surp2_exactness(theta, alpha, freeze_round, seed=0, n=1, m=7):
    world, sem, s1, l1, l0a = freeze_state(theta, alpha, freeze_round, seed, n, m)
    utts = list(sem.utterances)
    ctx0 = ScoreContext(TS, s1, world, sem, l1, utts[0], l0_theta=l0a)
    p, lp = ctx0.P_prior, ctx0.log_P_prior
    s_u = -lp - float(-(p @ lp))                       # surp2 at each utterance
    exact = float(p @ s_u**2 - (p @ s_u)**2)
    claimed = float(p @ np.array([
        compute_surp2(ScoreContext(TS, s1, world, sem, l1, u, l0_theta=l0a))[1]
        for u in utts]))
    return abs(claimed - exact)

errs = [surp2_exactness(th, a, fr, seed=fr)
        for a in [1.0, 3.0, 10.0] for th in [0.1, 0.5, 0.9] for fr in [2, 50]]
print(f"max |E[σ²_surp2] − exact| over {len(errs)} states : {max(errs):.2e}")

### Is the correction an exact identity?

Because the closed form is cheap, the identity can be checked directly at many states
rather than sampled. Writing $K$ for the correction term, the claim is

$$\mathrm{Var}_u[s(u)]\;=\;\mathbb{E}_u[\sigma^2_{\text{naive}}(u)]-K .$$

In [ ]:
def identity_check(theta, alpha, freeze_round, seed=0, n=1, m=7):
    world, sem, s1, l1, l0a = freeze_state(theta, alpha, freeze_round, seed, n, m)
    utts = list(sem.utterances)
    ctx0 = ScoreContext(TS, s1, world, sem, l1, utts[0], l0_theta=l0a)
    W, B = _weighting_matrix("1", ctx0), ctx0.B_matrix
    p = np.asarray(_p_pred("1", ctx0), dtype=float)
    s_u = (W*B).sum(axis=0)
    K = float(p @ np.maximum(np.sum(W*B**2, axis=0) - np.sum(W*B, axis=0)**2, 0.0))
    vn_u = np.array([sus1(ScoreContext(TS, s1, world, sem, l1, u, l0_theta=l0a))[1]
                     for u in utts])
    exact = float(p @ s_u**2 - (p @ s_u)**2)
    vc_u = np.array([sus1(ScoreContext(TS, s1, world, sem, l1, u, l0_theta=l0a))[2]
                     for u in utts])
    return {"theta": theta, "alpha": alpha, "round": freeze_round, "exact": exact,
            "E_naive_minus_K": float(p @ vn_u) - K,                # the identity
            "E_corrected": float(p @ vc_u),                        # what the code returns
            "E_clipped_old": float(p @ np.maximum(vn_u - K, 0.0)), # the old buggy rule
            "clipped_mass": float(p[vn_u < K].sum())}

rows = [identity_check(th, a, fr, seed=fr)
        for a in [1.0, 2.0, 3.0, 5.0, 10.0, 20.0]
        for th in [0.1, 0.3, 0.5, 0.7, 0.9]
        for fr in [2, 10, 50, 120]]
I = pd.DataFrame(rows)
I["err_identity"] = (I.E_naive_minus_K - I.exact).abs()
I["err_code"]     = (I.E_corrected     - I.exact).abs()
I["err_oldclip"]  = (I.E_clipped_old   - I.exact).abs()

print(f"states checked                            : {len(I)}")
print(f"max |E[σ²_naive] − K − exact|  (identity) : {I.err_identity.max():.2e}")
print(f"max |E[σ²_corrected] − exact|  (the code) : {I.err_code.max():.2e}")
print(f"max |old per-u clip − exact|   (the bug)  : {I.err_oldclip.max():.2e}")
print(f"\nstates where the identity holds to 1e-12  : {100*(I.err_identity < 1e-12).mean():.1f}%")
print(f"states where the code matches   to 1e-12  : {100*(I.err_code     < 1e-12).mean():.1f}%")
print(f"states the old clip would have broken     : {100*(I.err_oldclip >= 1e-12).mean():.1f}%")
print(f"\naffected states carried this mean clipped mass, by α:")
print(I[I.clipped_mass > 0].groupby("alpha").clipped_mass.mean().round(3).to_string())

**Verdict C — the correction is an exact identity. A clipping bug used to break it; it is now fixed.**

$\mathrm{Var}_u[s(u)]=\mathbb{E}_u[\sigma^2_{\text{naive}}(u)]-K$ holds to machine
precision at **every** state tested. The derivation is right, and `naive` alone is
genuinely biased high — by ~10 % at α = 1 — so the correction is doing real work.

The original implementation applied `max(var_naive - correction, 0)` **per utterance**,
but $K$ is a single constant for the state, so any utterance with
$\sigma^2_{\text{naive}}(u) < K$ was clipped upward. That added the clipped mass back and
inflated $\mathbb{E}_u[\sigma^2_{\text{corr}}]$ above the true variance — by up to 8.9 %
at α = 1, θ\* = 0.1, where the clipped utterances carried 60 % of the probability mass.

`scores.py` now returns the **unclipped** difference; individual rounds may be negative,
and `SequentialTest` clips the running *average* before the square root, which is the
correct place to enforce non-negativity. The cells above are kept as a regression guard —
`E[σ²_corrected]` should now match `exact` at every state, and the two error columns
should agree.

In [ ]:
w = identity_check(0.1, 1.0, 120, seed=120)
print("worst case in the grid — θ*=0.1, α=1, round 120")
print(f"  true Var[s]                          : {w['exact']:.6f}")
print(f"  E[σ²_naive] − K   (identity)         : {w['E_naive_minus_K']:.6f}")
print(f"  E[σ²_corrected]   (current code)     : {w['E_corrected']:.6f}")
print(f"  mass on utterances with σ²_naive < K : {w['clipped_mass']:.3f}")
print(f"  what the OLD per-u clip would give   : {w['E_clipped_old']:.6f} "
      f"(+{100*(w['E_clipped_old']/w['exact']-1):.1f}%)")

## D. Does the $1/t$ shrinkage hold?

$\mathrm{Var}[\mathrm{Sus}(t)]=\bar\sigma^2(t)/t$ presumes the per-round scores are
uncorrelated. Test both halves: the scaling itself, and the autocorrelation directly.

In [ ]:
d = load_cell(cid).sort_values(["sim_id", "round"])
d["Sus"]    = d.groupby("sim_id").sus1_score.cumsum()/d["round"]
d["sbar2"]  = d.groupby("sim_id").sus1_sigma2_naive.cumsum()/d["round"]
sc = d.groupby("round").apply(
    lambda g: pd.Series({"var_emp": g.Sus.var(ddof=1), "theory": g.sbar2.mean()/g.name}),
    include_groups=False)

n_r = int(d["round"].max())
Z = (d.sus1_score/np.sqrt(d.sus1_sigma2_naive.clip(lower=1e-300))).values.reshape(-1, n_r)
acf = [float(np.corrcoef(Z[:, :-k].ravel(), Z[:, k:].ravel())[0, 1]) for k in range(1, 11)]
ci = 1.96/np.sqrt(Z.size)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

ax = axes[0]
ax.plot(sc.index, sc.var_emp, color=C["naive"], label="empirical Var[Sus(t)]")
ax.plot(sc.index, sc.theory, color=C["corrected"], ls="--", label="theory  σ̄²(t)/t")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("round"); ax.set_ylabel("Var[Sus(t)]")
ax.set_title("D1. Running-mean shrinkage", color=INK, loc="left")
ax.legend(loc="lower left")

ax = axes[1]
ax.bar(range(1, 11), acf, color=C["naive"], width=.55)
ax.axhspan(-ci, ci, color=GRIDC, alpha=.8, zorder=0, label="95% band under independence")
ax.axhline(0, color=INK, lw=1.0)
ax.set_xlabel("lag"); ax.set_ylabel("autocorrelation of z-scores"); ax.set_xticks(range(1, 11))
ax.set_title("D2. Per-round score autocorrelation", color=INK, loc="left")
ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

print(f"median Var_emp/theory (rounds ≥ 20) : {(sc.var_emp/sc.theory).loc[20:].median():.3f}")
print(f"max |autocorrelation| over lags 1-10: {max(abs(a) for a in acf):.4f}   (95% band ±{ci:.4f})")

**Verdict D — the $1/t$ scaling is sound.** The largest autocorrelation over lags 1–10
is ≈ 0.013 — marginally outside the ±1.96/√N band, but far too small to matter at
N ≈ 30,000 (it would need to be orders of magnitude larger to bend the $1/t$ law), and
the empirical curve tracks $1/t$ rather than a slower law. The residual offset from 1 is
inherited from the per-round bias in §B, not from a broken CLT. This corroborates the
Group-1 finding that the FPR problem is multiple testing, not variance mis-scaling.

## 6. The `surp1` anomaly

`surp1` was dropped after Group 1 measured its variance ratio at ≈ 0.46. It is absent
from the full sweep, so this uses the Group-1 raw trajectories (α = 3, null only).

In [ ]:
g1 = pd.read_parquet(GROUP1/"raw_trajectories.parquet")
g1 = g1[g1.psi == "inf"]
rows = []
for (sc_name, t), g in g1.groupby(["score_type", "round"], observed=True):
    rows.append({"score": sc_name, "round": t, "ratio": g.score.var(ddof=1)/g.variance.mean()})
S = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7.2, 3.2))
for name, col in [("surp1", "exact"), ("surp2", "surp2"), ("sus", "naive")]:
    s = S[S.score == name].set_index("round").ratio.rolling(5, min_periods=1).mean()
    ax.plot(s.index, s, color=C[col], label=name)
ref_line(ax)
ax.set_xlabel("round"); ax.set_ylabel("Var_emp / E[σ²]"); ax.set_ylim(0, 1.4)
ax.set_title("6. Group-1 calibration, null, α=3 (5-round rolling mean)", color=INK, loc="left")
ax.legend(loc="lower right", ncols=3)
plt.tight_layout(); plt.show()

print(S.groupby("score").ratio.describe()[["mean", "50%", "min", "max"]].round(3).to_string())

`surp1` sits at ~0.46 — its claimed variance is about **twice** the empirical one.
The structural reason is that `surp1` uses the observed utterance twice: once as the
scored value, once inside the conditioning that produced $P_{\text{post}}$
(see the comment at `rsa/detection/scores.py:224`). The observed $u$ is therefore not
a fresh draw from the distribution its variance is computed against.

## 7. Verdicts

In [ ]:
verdicts = pd.DataFrame([
    ("A. E[s] = 0",                     "sus_1, surp2", "PASS",
     "exact to 1e-16; empirical excursions at chance"),
    ("B. σ² vs empirical (surp2)",      "surp2",        "PASS",
     "ratio 0.96-1.00 across all 99 null cells"),
    ("B. σ² vs empirical (naive)",      "sus_1",        "FAIL",
     "overstates by ~30% for α ≤ 3 (ratio 0.69-0.78)"),
    ("B. σ² vs empirical (corrected)",  "sus_1",        "PASS",
     "0.90-1.01 across all α; inert (= naive) above α≈10"),
    ("B'. score has usable signal",     "sus_1",        "FAIL α≥10",
     "degenerate rounds: 0% at α≤5, 10% α=10, 34% α=20"),
    ("C. replay at a frozen state",     "surp2",        "PASS",
     "σ² identical to exact (1e-16) at every state"),
    ("C. replay at a frozen state",     "sus_1 naive",  "FAIL",
     "biased high — ~10% above replayed variance at α=1"),
    ("C. LTV identity",                 "sus_1 corr.",  "PASS",
     "E[σ²_naive]−K = Var[s] to 1e-16 at every state"),
    ("C. identity as implemented",      "sus_1 corr.",  "PASS (fixed)",
     "per-u clip removed; code now matches exact at every state"),
    ("D. 1/t shrinkage",                "sus_1",        "PASS",
     "max |autocorr| 0.013 over lags 1-10; negligible"),
    ("surp1 variance",                  "surp1",        "FAIL",
     "ratio ≈0.46 — data used twice; σ² ~2x too large"),
], columns=["check", "score", "verdict", "evidence"])
print(verdicts.to_string(index=False))

### What this means

1. **`surp2` needs no fix.** Its variance is exact by construction and empirically confirmed.
2. **The law-of-total-variance correction is right.** $\mathrm{Var}_u[s(u)] =
   \mathbb{E}_u[\sigma^2_{\text{naive}}(u)] - K$ holds to machine precision at every state
   tested. `naive` on its own is genuinely biased high, so the correction is not cosmetic.
3. **A real bug in how it was applied — now fixed.** `scores.py` used to clip
   `max(var_naive − correction, 0)` **per utterance**, but $K$ is one constant for the
   state. Utterances with $\sigma^2_{\text{naive}}(u) < K$ were clipped upward, inflating
   $\mathbb{E}_u[\sigma^2_{\text{corr}}]$ by up to ~9 % at α = 1. It now returns the
   unclipped difference and `SequentialTest` floors the running average instead.
4. **The fix raises low-α FPR slightly, as predicted.** On identical utterance streams
   (300 sims × 150 rounds) it is inert wherever nothing clipped — α = 3 is bit-identical —
   and at α = 1, θ\* ∈ {0.1, 0.9}, where the clip bit ~51 % of rounds, FPR moves up by
   0–3 pp and TPR never drops. Small, but the sign is right and it is now unbiased.
5. **Cap α at 10 for a different reason than calibration.** Past α ≈ 10 the score itself
   degenerates on a growing share of rounds: the speaker is deterministic enough that
   there is no excess surprise left to measure. Detection results above α ≈ 10 are
   reporting on a dead signal, not a miscalibrated one.
6. **The FPR problem is still multiple testing.** §D rules out variance mis-scaling again,
   independently of Group 1. Per-round variance error makes the test *conservative*, so it
   cannot explain an inflated FPR.
7. **`surp1` should stay retired** unless the double-use of $u$ is redefined away.

**Next experiment.** Under the **model null** the corrected variance is now exact, yet §B —
measured under the **true sampling law** — still sits below 1 at low α. That leftover is the
listener's belief lagging $\theta^\star$, not a formula error. Probe it by freezing the
belief at the truth ($L_1(\theta)=\delta_{\theta^\star}$) and re-measuring the §B ratio: if
it snaps to 1 across α, the remedy is a warm-up rather than more algebra.

*Note on the stored data.* Everything in §B reads
`results/full_sweep/`, which was generated **before** the clip fix. Those parquets still
carry the old inflated `sus1_sigma2_corrected` at α ≤ 2. The §B conclusions are unaffected
(the clip is inert at the α where the naive/corrected gap matters), but a re-run is needed
before quoting low-α corrected numbers from that dataset.